In [ ]:
# ==========================================================
# PATIENT DROPOUT EVENT FORECASTING — FINAL PIPELINE
# Clinical Trial Studies: Phase 1 to Phase 4
# ==========================================================

# ---------- 1. SETUP ----------
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import re
import os
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import joblib

# ---------- 2. PROJECT CONFIGURATION ----------
project_path = "/content/drive/MyDrive/Saama/patient-dropout-event-forecasting/patient-dropout-event-forecasting"
phases = ["Phase1", "Phase2", "Phase3", "Phase4"]

TEXT_COLS = [
    "brief_summary/textblock",
    "eligibility/criteria/textblock",
    "brief_title",
    "intervention/description",
    "intervention/intervention_name",
    "condition",
]

MISSING_THRESHOLD = 90
CARDINALITY_THRESHOLD = 30
TFIDF_MAX_FEATURES = 150

BEST_MODEL_PER_PHASE = {
    "Phase1": "RandomForest",
    "Phase2": "RandomForest",
    "Phase3": "GradientBoosting",
    "Phase4": "GradientBoosting",
}

FINAL_MODELS_DIR = f"{project_path}/final_models"
os.makedirs(FINAL_MODELS_DIR, exist_ok=True)

# ---------- 3. PREPROCESSING FUNCTIONS ----------
def preprocess_phase(phase_name):
    X_train = pd.read_csv(f"{project_path}/{phase_name}/train_x.csv")
    y_train = pd.read_csv(f"{project_path}/{phase_name}/train_y.csv")
    X_test = pd.read_csv(f"{project_path}/{phase_name}/test_x.csv")
    y_test = pd.read_csv(f"{project_path}/{phase_name}/test_y.csv")

    id_col = "Unnamed: 0"
    X_train = X_train.drop(columns=[id_col])
    X_test = X_test.drop(columns=[id_col])
    if id_col in y_train.columns:
        y_train = y_train.drop(columns=[id_col])
    if id_col in y_test.columns:
        y_test = y_test.drop(columns=[id_col])

    y_train = y_train.rename(columns={"droupout_rate": "dropout_rate"})
    y_test = y_test.rename(columns={"droupout_rate": "dropout_rate"})

    missing_pct = (X_train.isnull().sum() / len(X_train)) * 100
    cols_to_drop = missing_pct[missing_pct > MISSING_THRESHOLD].index.tolist()
    X_train = X_train.drop(columns=cols_to_drop)
    X_test = X_test.drop(columns=cols_to_drop)

    text_cols_present = [c for c in TEXT_COLS if c in X_train.columns]
    X_train = X_train.drop(columns=text_cols_present)
    X_test = X_test.drop(columns=text_cols_present)

    num_cols = X_train.select_dtypes(include=["float64", "int64"]).columns.tolist()
    cat_cols = X_train.select_dtypes(include=["object"]).columns.tolist()

    if num_cols:
        num_imputer = SimpleImputer(strategy="median")
        X_train[num_cols] = num_imputer.fit_transform(X_train[num_cols])
        X_test[num_cols] = num_imputer.transform(X_test[num_cols])
    if cat_cols:
        cat_imputer = SimpleImputer(strategy="most_frequent")
        X_train[cat_cols] = cat_imputer.fit_transform(X_train[cat_cols])
        X_test[cat_cols] = cat_imputer.transform(X_test[cat_cols])

    cardinality = X_train[cat_cols].nunique()
    low_card_cols = cardinality[cardinality <= CARDINALITY_THRESHOLD].index.tolist()
    high_card_cols = cardinality[cardinality > CARDINALITY_THRESHOLD].index.tolist()

    for col in high_card_cols:
        freq_map = X_train[col].value_counts(normalize=True)
        X_train[col + "_freq"] = X_train[col].map(freq_map)
        X_test[col + "_freq"] = X_test[col].map(freq_map).fillna(0)
    X_train = X_train.drop(columns=high_card_cols)
    X_test = X_test.drop(columns=high_card_cols)

    X_train = pd.get_dummies(X_train, columns=low_card_cols)
    X_test = pd.get_dummies(X_test, columns=low_card_cols)
    X_train, X_test = X_train.align(X_test, join="left", axis=1, fill_value=0)

    return X_train, X_test, y_train, y_test


def fix_age_columns(X_train, X_test, phase_name):
    raw_train = pd.read_csv(f"{project_path}/{phase_name}/train_x.csv")
    raw_test = pd.read_csv(f"{project_path}/{phase_name}/test_x.csv")

    def parse_age(val):
        if pd.isna(val):
            return np.nan
        match = re.search(r"(\d+)", str(val))
        return float(match.group(1)) if match else np.nan

    for col, freq_col in [
        ("eligibility/minimum_age", "eligibility/minimum_age_freq"),
        ("eligibility/maximum_age", "eligibility/maximum_age_freq"),
    ]:
        if freq_col in X_train.columns:
            train_age = raw_train[col].apply(parse_age)
            test_age = raw_test[col].apply(parse_age)
            median_age = train_age.median()
            train_age = train_age.fillna(median_age)
            test_age = test_age.fillna(median_age)

            X_train = X_train.drop(columns=[freq_col])
            X_test = X_test.drop(columns=[freq_col])
            X_train[col.split("/")[-1]] = train_age.values
            X_test[col.split("/")[-1]] = test_age.values

    return X_train, X_test

# ---------- 4. TF-IDF FEATURE ENGINEERING ----------
def add_tfidf_features(X_train, X_test, phase_name):
    raw_train = pd.read_csv(f"{project_path}/{phase_name}/train_x.csv")
    raw_test = pd.read_csv(f"{project_path}/{phase_name}/test_x.csv")

    def combine_text(df):
        parts = [df[c].fillna("").astype(str) for c in TEXT_COLS if c in df.columns]
        return parts[0].str.cat(parts[1:], sep=" ") if len(parts) > 1 else parts[0]

    train_text = combine_text(raw_train)
    test_text = combine_text(raw_test)

    vectorizer = TfidfVectorizer(
        max_features=TFIDF_MAX_FEATURES,
        stop_words="english",
        min_df=3,
        ngram_range=(1, 1)
    )
    train_tfidf = vectorizer.fit_transform(train_text).toarray()
    test_tfidf = vectorizer.transform(test_text).toarray()

    tfidf_cols = [f"tfidf_{w}" for w in vectorizer.get_feature_names_out()]
    train_tfidf_df = pd.DataFrame(train_tfidf, columns=tfidf_cols, index=X_train.index)
    test_tfidf_df = pd.DataFrame(test_tfidf, columns=tfidf_cols, index=X_test.index)

    X_train_out = pd.concat([X_train.reset_index(drop=True), train_tfidf_df.reset_index(drop=True)], axis=1)
    X_test_out = pd.concat([X_test.reset_index(drop=True), test_tfidf_df.reset_index(drop=True)], axis=1)

    return X_train_out, X_test_out, vectorizer

# ---------- 5. TRAIN, VALIDATE, AND SAVE — ALL PHASES ----------
final_results = {}

for phase in phases:
    print(f"\n{'='*60}\nProcessing {phase}\n{'='*60}")

    X_train, X_test, y_train, y_test = preprocess_phase(phase)
    X_train, X_test = fix_age_columns(X_train, X_test, phase)

    nunique = X_train.nunique()
    zero_var_cols = nunique[nunique <= 1].index.tolist()
    X_train = X_train.drop(columns=zero_var_cols)
    X_test = X_test.drop(columns=zero_var_cols)

    X_train, X_test, vectorizer = add_tfidf_features(X_train, X_test, phase)

    y_train_target = y_train["dropout_rate"]
    y_test_target = y_test["dropout_rate"]

    if BEST_MODEL_PER_PHASE[phase] == "RandomForest":
        model = RandomForestRegressor(n_estimators=300, max_depth=12, random_state=42, n_jobs=-1)
    else:
        model = GradientBoostingRegressor(n_estimators=300, max_depth=4, learning_rate=0.05, random_state=42)

    cv_scores = cross_val_score(model, X_train, y_train_target, cv=5, scoring="r2")
    print(f"Cross-validation R2 scores (5 splits): {np.round(cv_scores, 4)}")
    print(f"Average: {cv_scores.mean():.4f}  |  Std deviation: {cv_scores.std():.4f}")

    model.fit(X_train, y_train_target)
    preds = model.predict(X_test)
    test_r2 = r2_score(y_test_target, preds)
    test_mae = mean_absolute_error(y_test_target, preds)
    test_rmse = np.sqrt(mean_squared_error(y_test_target, preds))
    print(f"Test set performance: R2={test_r2:.4f}, MAE={test_mae:.4f}, RMSE={test_rmse:.4f}")

    joblib.dump(model, f"{FINAL_MODELS_DIR}/{phase}_model.pkl")
    joblib.dump(vectorizer, f"{FINAL_MODELS_DIR}/{phase}_tfidf_vectorizer.pkl")
    print(f"Saved model and vectorizer for {phase} to {FINAL_MODELS_DIR}")

    final_results[phase] = {
        "model_type": BEST_MODEL_PER_PHASE[phase],
        "cv_mean_r2": round(cv_scores.mean(), 4),
        "cv_std": round(cv_scores.std(), 4),
        "test_r2": round(test_r2, 4),
        "test_mae": round(test_mae, 4),
        "test_rmse": round(test_rmse, 4),
    }

# ---------- 6. FINAL RESULTS SUMMARY ----------
print("\n" + "="*60)
print("FINAL RESULTS SUMMARY")
print("="*60)
summary_df = pd.DataFrame(final_results).T
print(summary_df.to_string())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).

Processing Phase1
Cross-validation R2 scores (5 splits): [0.4562 0.4241 0.489  0.4453 0.4235]
Average: 0.4476  |  Std deviation: 0.0242
Test set performance: R2=0.5473, MAE=0.1437, RMSE=0.2131
Saved model and vectorizer for Phase1 to /content/drive/MyDrive/Saama/patient-dropout-event-forecasting/patient-dropout-event-forecasting/final_models

Processing Phase2
Cross-validation R2 scores (5 splits): [0.2435 0.2525 0.2782 0.2426 0.2413]
Average: 0.2516  |  Std deviation: 0.0138
Test set performance: R2=0.2597, MAE=0.2083, RMSE=0.2775
Saved model and vectorizer for Phase2 to /content/drive/MyDrive/Saama/patient-dropout-event-forecasting/patient-dropout-event-forecasting/final_models

Processing Phase3
Cross-validation R2 scores (5 splits): [0.3021 0.3236 0.3022 0.2486 0.2936]
Average: 0.2940  |  Std deviation: 0.0248
Test set performance: R2=0.3047, MAE=0.1701,